## Bronze Layer: Ingestion & Enrichment

Phase 1 pipeline — ingests raw CSVs into Bronze parquet, then scrapes TMDB metadata for the top 500 movies.

### 1. Start a SparkSession

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Bronze") \
    .master("local[*]") \
    .getOrCreate()

### 2. Define schemas

In [ ]:
from pyspark.sql.types import *

ratings_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("rating",       FloatType(),    True),
    StructField("timestamp",    LongType(),     True),
])

movies_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("title",    StringType(),   True),
    StructField("genres",   StringType(),   True),
])

links_schema = StructType([
    StructField("movieId",  IntegerType(),  True),
    StructField("imdbId",   IntegerType(),  True),
    StructField("tmdbId",   IntegerType(),  True),
])

tags_schema = StructType([
    StructField("userId",       IntegerType(),  True),
    StructField("movieId",      IntegerType(),  True),
    StructField("tag",          StringType(),   True),
    StructField("timestamp",    LongType(),     True),
])

### 3. Read each CSV

In [ ]:
df_ratings = spark.read.csv("ml-32m/ratings.csv", header=True, schema=ratings_schema)
df_movies  = spark.read.csv("ml-32m/movies.csv",  header=True, schema=movies_schema)
df_links   = spark.read.csv("ml-32m/links.csv",   header=True, schema=links_schema)
df_tags    = spark.read.csv("ml-32m/tags.csv",    header=True, schema=tags_schema)

### 4. Add ingestion metadata

In [ ]:
from pyspark.sql.functions import current_timestamp, lit

def add_metadata(df, source_name):
    return df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source_file", lit(source_name))

### 5. Write to bronze/ as parquet

In [ ]:
add_metadata(df_ratings, "ratings.csv") \
    .write.mode("overwrite").parquet("bronze/ratings")

add_metadata(df_movies, "movies.csv") \
    .write.mode("overwrite").parquet("bronze/movies")

add_metadata(df_links, "links.csv") \
    .write.mode("overwrite").parquet("bronze/links")

add_metadata(df_tags, "tags.csv") \
    .write.mode("overwrite").parquet("bronze/tags")

### 6. Sanity checks

In [ ]:
for name, df in [("ratings", df_ratings), ("movies", df_movies),
                  ("links", df_links), ("tags", df_tags)]:
    print(f"\n=== {name} ===")
    print(f"Rows: {df.count()}")
    df.printSchema()
    df.show(3)

---

## TMDB Enrichment (Top 500 Movies)

### 7. Identify the top 500 movies

In [ ]:
from pyspark.sql.functions import col, count, avg, round

top_500 = (
    df_ratings
    .groupBy("movieId")
    .agg(count("*").alias("rating_count"), round(avg("rating"), 2).alias("avg_rating"))
    .orderBy(col("rating_count").desc())
    .limit(500)
)

movies_to_scrape = (
    top_500
    .join(df_links, "movieId")
    .join(df_movies, "movieId")
    .select("movieId", "title", "tmdbId", "imdbId", "rating_count", "avg_rating")
    .filter(col("tmdbId").isNotNull())
    .orderBy(col("avg_rating").desc())
    .collect()
)

for row in movies_to_scrape[:10]:
    print(f"{row.movieId:<10}{row.title:<50} | IMDB: {row.imdbId: <10} | TMDB: {row.tmdbId: <10} | Ratings: {row.rating_count: <8} | Avg: {row.avg_rating}")

print(f"\nTotal movies: {len(movies_to_scrape)}")

### 8. TMDB scraper utility

In [ ]:
import os
import time
import requests
from dotenv import load_dotenv

load_dotenv()

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {os.getenv('TMDB_BEARER_TOKEN')}",
    "accept": "application/json",
})


def fetch_movie(tmdb_id: int) -> dict | None:
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}?append_to_response=credits"
    while True:
        resp = session.get(url, timeout=10)
        if resp.status_code == 429:
            time.sleep(int(resp.headers.get("Retry-After", 2)))
            continue
        if resp.status_code != 200:
            print(f"  WARN: tmdb_id={tmdb_id} -> {resp.status_code}")
            return None
        return resp.json()


def extract(raw: dict, movie_id: int) -> dict:
    poster = raw.get("poster_path")
    return {
        "movieId": movie_id,
        "tmdbId": raw.get("id"),
        "title": raw.get("title"),
        "directors": [m["name"] for m in raw.get("credits", {}).get("crew", []) if m.get("job") == "Director"],
        "budget": raw.get("budget"),
        "revenue": raw.get("revenue"),
        "runtime": raw.get("runtime"),
        "release_date": raw.get("release_date"),
        "poster_url": f"https://image.tmdb.org/t/p/w500{poster}" if poster else None,
        "overview": raw.get("overview"),
        "vote_average": raw.get("vote_average"),
        "original_language": raw.get("original_language"),
    }

### 9. Scrape & save to parquet

In [ ]:
import json

results = []
total = len(movies_to_scrape)

for i, row in enumerate(movies_to_scrape):
    print(f"[{i+1}/{total}] tmdbId={row.tmdbId} ({row.title})")
    raw = fetch_movie(row.tmdbId)
    if raw:
        results.append(extract(raw, row.movieId))
    time.sleep(0.05)

print(f"\nScraped {len(results)}/{total} movies.")

# Save JSON for sharing
json_path = "bronze/enrichment/scraped_metadata.json"
os.makedirs(os.path.dirname(json_path), exist_ok=True)
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"JSON saved to {json_path}")

# Save parquet for pipeline
enrichment_schema = StructType([
    StructField("movieId",           IntegerType()),
    StructField("tmdbId",            IntegerType()),
    StructField("title",             StringType()),
    StructField("directors",         ArrayType(StringType())),
    StructField("budget",            LongType()),
    StructField("revenue",           LongType()),
    StructField("runtime",           IntegerType()),
    StructField("release_date",      StringType()),
    StructField("poster_url",        StringType()),
    StructField("overview",          StringType()),
    StructField("vote_average",      FloatType()),
    StructField("original_language", StringType()),
])

df_enrichment = (
    spark.createDataFrame(results, enrichment_schema)
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", lit("tmdb_api"))
)

df_enrichment.write.mode("overwrite").parquet("bronze/enrichment/parquet")

print(f"Done. {df_enrichment.count()} rows written to bronze/enrichment/parquet")
df_enrichment.show(5, truncate=False)